In [0]:
# Imports

from pyspark.sql.functions import (
    col, round as spark_round,
    first, current_timestamp, date_format
)
from delta.tables import DeltaTable

print("Imports successful")

In [0]:
# Read Bronze food and dim_country
df_bronze_food = spark.table("workspace.default.bronze_world_bank_food")
df_dim_country = spark.table("workspace.default.dim_country")

print(f"Bronze food rows: {df_bronze_food.count()}")
print(f"dim_country rows: {df_dim_country.count()}")

# what we are about to pivot
print("\nBronze food sample — notice indicator_name column:")
display(
    df_bronze_food
    .select("country_code", "year", "indicator_name", "value")
    .filter(col("country_code") == "KE")
    .orderBy("year", "indicator_name")
    .limit(15)
)

In [0]:
# Pivot long to wide format

df_pivoted = (
    df_bronze_food
    .groupBy("country_code", "year")
    .pivot(
        "indicator_name",
        ["undernourishment_pct", "stunting_pct", "underweight_pct"]
    )
    .agg(first("value"))
)

# Round all three indicator columns to 2 decimal places
df_pivoted = (
    df_pivoted
    .withColumn("undernourishment_pct", spark_round(col("undernourishment_pct"), 2))
    .withColumn("stunting_pct",         spark_round(col("stunting_pct"),         2))
    .withColumn("underweight_pct",      spark_round(col("underweight_pct"),       2))
)

print(f"Pivoted rows: {df_pivoted.count()}")
display(
    df_pivoted
    .filter(col("country_code") == "KE")
    .orderBy("year")
)

In [0]:
# Join against dim_country

df_silver_food = (
    df_pivoted
    .join(
        df_dim_country.select("country_code", "country_iso3", "country_name", "region"),
        on="country_code",
        how="inner"
    )
    .select(
        "country_code",
        "country_iso3",
        "country_name",
        "region",
        "year",
        "undernourishment_pct",
        "stunting_pct",
        "underweight_pct"
    )
)

df_silver_food = df_silver_food.withColumn(
    "updated_at",
    date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss")
)

print(f"Silver food rows after join: {df_silver_food.count()}")

display(
    df_silver_food
    .filter(col("country_code") == "KE")
    .orderBy("year")
)

In [0]:
# First-time write
TABLE_NAME = "workspace.default.silver_fact_food"

(df_silver_food.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLE_NAME)
)

print(f"Silver food table created: {TABLE_NAME}")
print(f"Row count: {spark.table(TABLE_NAME).count()}")

In [0]:
# MERGE INTO

TABLE_NAME = "workspace.default.silver_fact_food"

delta_target = DeltaTable.forName(spark, TABLE_NAME)

(
    delta_target.alias("target")
    .merge(
        df_silver_food.alias("source"),
        "target.country_code = source.country_code AND target.year = source.year"
    )
    .whenMatchedUpdate(set={
        "undernourishment_pct": "source.undernourishment_pct",
        "stunting_pct":         "source.stunting_pct",
        "underweight_pct":      "source.underweight_pct",
        "country_iso3":         "source.country_iso3",
        "country_name":         "source.country_name",
        "region":               "source.region",
        "updated_at":           "source.updated_at",
    })
    .whenNotMatchedInsert(values={
        "country_code":         "source.country_code",
        "country_iso3":         "source.country_iso3",
        "country_name":         "source.country_name",
        "region":               "source.region",
        "year":                 "source.year",
        "undernourishment_pct": "source.undernourishment_pct",
        "stunting_pct":         "source.stunting_pct",
        "underweight_pct":      "source.underweight_pct",
        "updated_at":           "source.updated_at",
    })
    .execute()
)

print("MERGE complete")
print(f"Row count after MERGE: {spark.table(TABLE_NAME).count()}")

In [0]:
# Verify

from pyspark.sql.functions import avg, min, max, isnan, when, count as spark_count

df_check = spark.table("workspace.default.silver_fact_food")

print(f"Total rows: {df_check.count()}")

display(
    df_check.groupBy("country_code", "country_name")
            .agg(
                spark_round(avg("undernourishment_pct"), 2).alias("avg_undernourishment"),
                spark_round(avg("stunting_pct"),         2).alias("avg_stunting"),
                spark_round(avg("underweight_pct"),      2).alias("avg_underweight"),
            )
            .orderBy("avg_undernourishment", ascending=False)
)

In [0]:
# Null check across all three indicators
display(
    df_check.select(
        spark_count(when(col("undernourishment_pct").isNull(), 1)).alias("nulls_undernourishment"),
        spark_count(when(col("stunting_pct").isNull(),         1)).alias("nulls_stunting"),
        spark_count(when(col("underweight_pct").isNull(),      1)).alias("nulls_underweight"),
    )
)